# 📦 Demand Forecasting & Inventory Optimization Engine
## Milestone 2: Forecast Model Development

---

**👥 Team Members:** Mark · Fady · Ahmed · Eman · Sama · Hadeer  
**📅 Date:** February 2026  
**🎯 Objective:** Train, evaluate, and compare multiple forecasting models — from classical statistical methods to modern ML approaches.

---

### 📋 Table of Contents
1. [Setup & Data Loading](#1.-Setup-&-Data-Loading)
2. [Train / Validation / Test Split](#2.-Train-/-Validation-/-Test-Split)
3. [Baseline Model (Naive / SARIMA)](#3.-Baseline-Model)
4. [Prophet Model](#4.-Prophet-Model)
5. [XGBoost Model](#5.-XGBoost-Model)
6. [Model Comparison & Evaluation](#6.-Model-Comparison-&-Evaluation)
7. [Best Model — Final Evaluation on Test Set](#7.-Best-Model-Final-Evaluation-on-Test-Set)
8. [Save Best Model](#8.-Save-Best-Model)

---
## 1. Setup & Data Loading

In [ ]:
# ── Core ──────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
import joblib
from pathlib import Path

# ── Visualization ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# ── Metrics ───────────────────────────────────────────────────────────────────
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ── Preprocessing ─────────────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ── XGBoost ───────────────────────────────────────────────────────────────────
from xgboost import XGBRegressor

# ── Prophet ───────────────────────────────────────────────────────────────────
try:
    from prophet import Prophet
    PROPHET_AVAILABLE = True
except ImportError:
    print('⚠️  Prophet not installed. Run: pip install prophet')
    PROPHET_AVAILABLE = False

# ── SARIMA ────────────────────────────────────────────────────────────────────
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT  = Path('..').resolve()
PROC_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR    = PROJECT_ROOT / 'src' / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

DATE_COL   = 'date'    # <-- UPDATE if different
TARGET_COL = 'demand'  # <-- UPDATE if different

print('✅ Setup complete!')

In [ ]:
# Load cleaned data from Milestone 1
df = pd.read_parquet(PROC_DATA_DIR / 'demand_cleaned.parquet')
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values(DATE_COL).reset_index(drop=True)

print(f'✅ Loaded: {df.shape}')
df[[DATE_COL, TARGET_COL]].tail(10)

---
## 2. Train / Validation / Test Split

> **Strategy:** Chronological split — never shuffle time-series data!

In [ ]:
# ── Split Ratios  (70% train / 15% val / 15% test) ───────────────────────────
n = len(df)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

df_train = df.iloc[:train_end].copy()
df_val   = df.iloc[train_end:val_end].copy()
df_test  = df.iloc[val_end:].copy()

print(f'📊 Split Summary:')
print(f'   Train : {len(df_train):>6,} rows  ({df_train[DATE_COL].min().date()} → {df_train[DATE_COL].max().date()})')
print(f'   Val   : {len(df_val):>6,} rows  ({df_val[DATE_COL].min().date()} → {df_val[DATE_COL].max().date()})')
print(f'   Test  : {len(df_test):>6,} rows  ({df_test[DATE_COL].min().date()} → {df_test[DATE_COL].max().date()})')

# Visualize split
fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(df_train[DATE_COL], df_train[TARGET_COL], label='Train',      color='#3498db', linewidth=0.8)
ax.plot(df_val[DATE_COL],   df_val[TARGET_COL],   label='Validation', color='#f39c12', linewidth=0.8)
ax.plot(df_test[DATE_COL],  df_test[TARGET_COL],  label='Test',       color='#e74c3c', linewidth=0.8)
ax.set_title('Train / Validation / Test Split', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Shared Evaluation Helper ─────────────────────────────────────────────────

def evaluate(y_true, y_pred, model_name='Model'):
    """Compute and print MAE, RMSE, MAPE, R² for a forecast."""
    y_true_arr = np.array(y_true)
    y_pred_arr = np.array(y_pred)
    mae  = mean_absolute_error(y_true_arr, y_pred_arr)
    rmse = np.sqrt(mean_squared_error(y_true_arr, y_pred_arr))
    mape = np.mean(np.abs((y_true_arr - y_pred_arr) / (y_true_arr + 1e-9))) * 100
    r2   = r2_score(y_true_arr, y_pred_arr)
    print(f'  [{model_name}]  MAE={mae:.2f}  RMSE={rmse:.2f}  MAPE={mape:.2f}%  R²={r2:.4f}')
    return {'model': model_name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'R2': r2}

results = []   # Will accumulate all model results
print('✅ evaluate() helper ready')

---
## 3. Baseline Model
### 3a. Naïve Forecast (Last Observed Value)

In [ ]:
# Naïve: predict the last training value for all validation steps
naive_pred = np.full(len(df_val), df_train[TARGET_COL].iloc[-1])
res = evaluate(df_val[TARGET_COL], naive_pred, model_name='Naïve Baseline')
results.append(res)

### 3b. SARIMA Model

In [ ]:
# 🔧 Tune (p,d,q) and (P,D,Q,s) for your data
# Hint: use ACF/PACF plots from Milestone 1 notebook
ts_train = df_train.set_index(DATE_COL)[TARGET_COL].resample('D').sum()

sarima_model = SARIMAX(
    ts_train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),  # weekly seasonality
    enforce_stationarity=False,
    enforce_invertibility=False
)
sarima_fit = sarima_model.fit(disp=False)
print(sarima_fit.summary())

In [ ]:
sarima_pred = sarima_fit.forecast(steps=len(df_val))
res = evaluate(df_val[TARGET_COL].values, sarima_pred.values, model_name='SARIMA')
results.append(res)

# Plot
fig, ax = plt.subplots(figsize=FIGSIZE_WIDE if 'FIGSIZE_WIDE' in dir() else (16,4))
ax.plot(df_val[DATE_COL].values, df_val[TARGET_COL].values, label='Actual',        color='#2c3e50')
ax.plot(df_val[DATE_COL].values, sarima_pred.values,         label='SARIMA Pred',   color='#e74c3c', linestyle='--')
ax.set_title('SARIMA Forecast vs Actual', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

---
## 4. Prophet Model

In [ ]:
if PROPHET_AVAILABLE:
    # Prophet needs columns named 'ds' and 'y'
    prophet_train = df_train[[DATE_COL, TARGET_COL]].rename(
        columns={DATE_COL: 'ds', TARGET_COL: 'y'}
    )
    prophet_val = df_val[[DATE_COL]].rename(columns={DATE_COL: 'ds'})

    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        changepoint_prior_scale=0.05
    )
    m.fit(prophet_train)

    forecast = m.predict(prophet_val)
    prophet_pred = forecast['yhat'].values
    prophet_pred = np.clip(prophet_pred, 0, None)  # demand can't be negative

    res = evaluate(df_val[TARGET_COL].values, prophet_pred, model_name='Prophet')
    results.append(res)

    # Prophet components plot
    fig = m.plot_components(forecast)
    plt.suptitle('Prophet Components', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print('⚠️  Skipping Prophet (not installed)')

---
## 5. XGBoost Model

In [ ]:
# ── Feature Engineering for XGBoost ─────────────────────────────────────────
def make_features(df_in, target_col, lags=[1,7,14,30], windows=[7,14,30]):
    df_out = df_in.copy()
    # Lag features
    for lag in lags:
        df_out[f'lag_{lag}'] = df_out[target_col].shift(lag)
    # Rolling window features
    for w in windows:
        df_out[f'roll_mean_{w}'] = df_out[target_col].shift(1).rolling(w).mean()
        df_out[f'roll_std_{w}']  = df_out[target_col].shift(1).rolling(w).std()
    return df_out.dropna()

# Build features on full train+val for safe lag computation
df_feat = make_features(df, TARGET_COL)

feature_cols = [c for c in df_feat.columns if c not in [DATE_COL, TARGET_COL]]
print(f'✅ Features ({len(feature_cols)}): {feature_cols}')

In [ ]:
# Re-do the split on feature-engineered dataframe
n_feat   = len(df_feat)
t_end_f  = int(n_feat * 0.70)
v_end_f  = int(n_feat * 0.85)

X_train = df_feat.iloc[:t_end_f][feature_cols]
y_train = df_feat.iloc[:t_end_f][TARGET_COL]
X_val   = df_feat.iloc[t_end_f:v_end_f][feature_cols]
y_val   = df_feat.iloc[t_end_f:v_end_f][TARGET_COL]
X_test  = df_feat.iloc[v_end_f:][feature_cols]
y_test  = df_feat.iloc[v_end_f:][TARGET_COL]

# Train XGBoost
xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50,
    eval_metric='rmse'
)

xgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

xgb_pred = xgb.predict(X_val)
xgb_pred = np.clip(xgb_pred, 0, None)

res = evaluate(y_val.values, xgb_pred, model_name='XGBoost')
results.append(res)
print(f'  Best iteration: {xgb.best_iteration}')

In [ ]:
# ── Feature Importance ────────────────────────────────────────────────────────
importance = pd.Series(xgb.feature_importances_, index=feature_cols).sort_values(ascending=False)

top_n = min(20, len(importance))
fig, ax = plt.subplots(figsize=(10, 6))
importance.head(top_n).plot(kind='barh', ax=ax, color='#3498db', edgecolor='white')
ax.invert_yaxis()
ax.set_title(f'Top {top_n} XGBoost Feature Importances', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'docs' / 'model_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Model Comparison & Evaluation

In [ ]:
results_df = pd.DataFrame(results).set_index('model')
print('\n📊 MODEL COMPARISON (Validation Set)')
print('='*60)
print(results_df.to_string())

# Highlight best model per metric
print('\n🏆 Best Model per Metric:')
for metric in ['MAE', 'RMSE', 'MAPE']:
    best = results_df[metric].idxmin()
    print(f'   {metric}: {best}  ({results_df.loc[best, metric]:.4f})')
for metric in ['R2']:
    best = results_df[metric].idxmax()
    print(f'   {metric}: {best}  ({results_df.loc[best, metric]:.4f})')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Model Comparison on Validation Set', fontsize=15, fontweight='bold')

colors = sns.color_palette('husl', len(results_df))

for ax, metric in zip(axes, ['MAE', 'RMSE', 'MAPE']):
    bars = ax.bar(results_df.index, results_df[metric], color=colors, edgecolor='white')
    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.set_xticklabels(results_df.index, rotation=15, ha='right')
    ax.bar_label(bars, fmt='%.2f', padding=3, fontsize=9)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'docs' / 'model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Best Model — Final Evaluation on Test Set

> **⚠️ Only run this ONCE after choosing the best model. Do NOT use test set for tuning!**

In [ ]:
# 🔧 Change BEST_MODEL to whichever scored best above
BEST_MODEL = 'XGBoost'  # Options: 'SARIMA', 'Prophet', 'XGBoost'

if BEST_MODEL == 'XGBoost':
    test_pred = np.clip(xgb.predict(X_test), 0, None)
    test_true = y_test.values
    test_dates = df_feat.iloc[v_end_f:][DATE_COL].values
elif BEST_MODEL == 'Prophet' and PROPHET_AVAILABLE:
    future = df_test[[DATE_COL]].rename(columns={DATE_COL: 'ds'})
    fc     = m.predict(future)
    test_pred = np.clip(fc['yhat'].values, 0, None)
    test_true = df_test[TARGET_COL].values
    test_dates = df_test[DATE_COL].values
elif BEST_MODEL == 'SARIMA':
    test_pred = sarima_fit.forecast(steps=len(df_test)).values
    test_true = df_test[TARGET_COL].values
    test_dates = df_test[DATE_COL].values

# Evaluate on test
test_res = evaluate(test_true, test_pred, model_name=f'{BEST_MODEL} (TEST)')

# Plot
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(test_dates, test_true,  label='Actual',       color='#2c3e50', linewidth=1.2)
ax.plot(test_dates, test_pred,  label=f'{BEST_MODEL} Forecast', color='#e74c3c', linewidth=1.2, linestyle='--')
ax.fill_between(test_dates,
    np.array(test_pred) * 0.9, np.array(test_pred) * 1.1,
    alpha=0.15, color='#e74c3c', label='±10% Band')
ax.set_title(f'{BEST_MODEL} — Final Test Set Evaluation', fontsize=14, fontweight='bold')
ax.legend()
ax.set_xlabel('Date')
ax.set_ylabel('Demand')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'docs' / 'model_test_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Save Best Model

In [ ]:
model_path = MODELS_DIR / f'{BEST_MODEL.lower()}_demand_forecast.pkl'

if BEST_MODEL == 'XGBoost':
    joblib.dump({'model': xgb, 'feature_cols': feature_cols}, model_path)
elif BEST_MODEL == 'Prophet' and PROPHET_AVAILABLE:
    joblib.dump(m, model_path)
elif BEST_MODEL == 'SARIMA':
    sarima_fit.save(model_path.with_suffix('.sarimax'))

print(f'✅ Model saved → {model_path}')
print(f'   Test MAE  : {test_res["MAE"]:.4f}')
print(f'   Test RMSE : {test_res["RMSE"]:.4f}')
print(f'   Test MAPE : {test_res["MAPE"]:.2f}%')
print(f'   Test R²   : {test_res["R2"]:.4f}')